In [9]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore', category=RuntimeWarning)

In [10]:
# =============================================================================
# CONFIGURATION  –  edit this cell only
# =============================================================================

MIN_FRAME      = 20
MAX_FRAME      = 100
OUTPUT_DIR     = 'Output__'
CHANNEL_SUFFIX = 'Ch1'     # set to '' if unused
CWD            = os.getcwd()

DATA_CONFIG = {
    'WT': {
        'path': 'WT',
        'drug_suffix': 'psi',
        'slices': {'data1': [1,2,3], 'data2': [1,2,3,4], 'data3': [1,2,3,4]},
    },
    'AV': {
        'path': 'Antagonist- Volinanserin',
        'drug_suffix': 'psi+antag',
        'slices': {'data1': [1,2,3,4], 'data2': [1,2,3,4]},
    },
    'IP': {
        'path': 'IP3R2 cKO',
        'drug_suffix': 'psi',
        'slices': {'data1': [2,4,5], 'data2': [1,2,3]},
    },
    'CE': {
        'path': 'CalEx',
        'drug_suffix': 'psi',
        'slices': {'data1': [1,2,3,4,5,6], 'data2': [1,2,3]},
    },
}

ALL_FEATURES = [
    "Curve - Max Df", "Curve - Max Dff",
    "Curve - dat AUC", "Curve - df AUC", "Curve - dff AUC",
    "Basic - Area", "Basic - Perimeter (only for 2D video)", "Basic - Circularity",
    "Curve - Duration of visualized event overlay",
    "Curve - Duration 50% to 50% based on averge dF/F",
    "Curve - Duration 10% to 10% based on averge dF/F",
    "Curve - Rising duration 10% to 90% based on averge dF/F",
    "Curve - Decaying duration 90% to 10% based on averge dF/F",
    "Network - number of events in the same location",
    "Network - number of events in the same location with similar size only",
    "Network - maximum number of events appearing at the same time",
]

ALPHA         = 0.05
CONTROL_GROUP = 'WT'
BIN_SIZES     = [5, 10, 20]   # frame bin widths for CSV exports

# Plotting
COLORS      = {'WT': '#4878CF', 'AV': '#6ACC65', 'IP': '#D65F5F', 'CE': '#B47CC7'}
GROUP_ORDER = list(DATA_CONFIG.keys())

# Short display labels for plots (avoids truncation of long feature names)
FEATURE_LABELS = {
    "Curve - Max Df":                                                    "Max ΔF",
    "Curve - Max Dff":                                                   "Max ΔF/F",
    "Curve - dat AUC":                                                   "AUC (raw)",
    "Curve - df AUC":                                                    "AUC (ΔF)",
    "Curve - dff AUC":                                                   "AUC (ΔF/F)",
    "Basic - Area":                                                      "Area",
    "Basic - Perimeter (only for 2D video)":                             "Perimeter",
    "Basic - Circularity":                                               "Circularity",
    "Curve - Duration of visualized event overlay":                      "Duration (overlay)",
    "Curve - Duration 50% to 50% based on averge dF/F":                 "Duration (50–50%)",
    "Curve - Duration 10% to 10% based on averge dF/F":                 "Duration (10–10%)",
    "Curve - Rising duration 10% to 90% based on averge dF/F":          "Rise time (10–90%)",
    "Curve - Decaying duration 90% to 10% based on averge dF/F":        "Decay time (90–10%)",
    "Network - number of events in the same location":                   "Co-location events",
    "Network - number of events in the same location with similar size only": "Co-location (same size)",
    "Network - maximum number of events appearing at the same time":     "Max simultaneous events",
}

def _feat_label(feat):
    """Return short display label for a feature name."""
    return FEATURE_LABELS.get(feat, feat.split(' - ')[-1][:40])

In [11]:
# =============================================================================
# DATA LOADING AND NORMALISATION
# =============================================================================

_DROP_COLS = {'Channel', 'Index', 'Curve - P Value on max Dff (-log10)', 'Curve - Decay tau'}
_META_COLS = {'Starting Frame', 'data_folder', 'slice_num'}


def _file_paths(config):
    """Yield (baseline_path, drug_path, washout_path, subfolder, slice_num)."""
    ch = f'_{CHANNEL_SUFFIX}' if CHANNEL_SUFFIX else ''
    for subfolder, nums in config['slices'].items():
        folder = os.path.join(CWD, OUTPUT_DIR, config['path'], subfolder)
        for n in nums:
            yield (
                os.path.join(folder, f'slice{n}_baseline_AQuA2{ch}.csv'),
                os.path.join(folder, f'slice{n}_{config["drug_suffix"]}_AQuA2{ch}.csv'),
                os.path.join(folder, f'slice{n}_washout_AQuA2{ch}.csv'),
                subfolder, n,
            )


def _load_file(path):
    """Load one AQuA2 CSV → tidy DataFrame filtered to [MIN_FRAME, MAX_FRAME]."""
    if not os.path.exists(path):
        return pd.DataFrame()
    df = pd.read_csv(path, header=None).set_index(0).T.reset_index(drop=True)
    df = df.drop(columns=list(_DROP_COLS & set(df.columns)), errors='ignore')
    df = df.apply(pd.to_numeric, errors='coerce')
    if 'Starting Frame' in df.columns:
        df = df[df['Starting Frame'].between(MIN_FRAME, MAX_FRAME)].copy()
    return df if not df.empty else pd.DataFrame()


def _normalise(df, medians):
    """
    Fold-change normalisation: divide each feature value by that feature's
    baseline median for this slice.  Metadata columns are passed through
    unchanged.  Zero or missing baseline medians produce NaN (not division
    by zero); any resulting Inf values are also replaced with NaN.
    """
    out = df.copy()
    for col in df.columns:
        if col in _META_COLS:
            continue
        m = medians.get(col, np.nan)
        out[col] = np.nan if (pd.isna(m) or m == 0) else df[col] / m
    return out.replace([np.inf, -np.inf], np.nan)


def load_group(group_name):
    """
    Load all slices for one group in a single pass.

    Returns (baseline_norm, drug_norm, washout_norm, baseline_raw, drug_raw, washout_raw).
    Each slice is read once; raw and normalised DataFrames are built together
    so the same baseline median is used for all three conditions of a slice.
    A slice is skipped entirely if any of its three condition files is missing
    or empty — partial triplets would break within-slice normalisation.
    """
    lists = {k: [] for k in ['b_norm','d_norm','w_norm','b_raw','d_raw','w_raw']}

    for bp, dp, wp, subfolder, snum in _file_paths(DATA_CONFIG[group_name]):
        b, d, w = _load_file(bp), _load_file(dp), _load_file(wp)
        if b.empty or d.empty or w.empty:
            if not b.empty:
                print(f'  Skipping {group_name}/{subfolder}/slice{snum}: incomplete triplet')
            continue

        # Baseline median used as the FC denominator for all three conditions
        meds = b.median(numeric_only=True).replace(0, np.nan)
        for raw, norm, key_r, key_n in [(b, _normalise(b, meds), 'b_raw', 'b_norm'),
                                         (d, _normalise(d, meds), 'd_raw', 'd_norm'),
                                         (w, _normalise(w, meds), 'w_raw', 'w_norm')]:
            for df, key in [(raw, key_r), (norm, key_n)]:
                df['data_folder'] = subfolder
                df['slice_num']   = snum
                lists[key].append(df)

    def _concat(key):
        return pd.concat(lists[key], ignore_index=True) if lists[key] else pd.DataFrame()

    if not lists['b_norm']:
        print(f'  WARNING: {group_name} – no valid slices loaded. '
              f'Check that files exist under Output__/{DATA_CONFIG[group_name]["path"]}/')

    return tuple(_concat(k) for k in ['b_norm','d_norm','w_norm','b_raw','d_raw','w_raw'])


# ── Saving ───────────────────────────────────────────────────────────────────

def _out_dir(group_name):
    p = os.path.join(CWD, OUTPUT_DIR, DATA_CONFIG[group_name]['path'])
    os.makedirs(p, exist_ok=True)
    return p


def save_csvs(base, drug, wash, group_name, tag):
    """Save baseline/drug/washout DataFrames as CSVs with a filename tag."""
    d = _out_dir(group_name)
    for df, cond in zip([base, drug, wash], ['baseline', 'drug', 'washout']):
        df.to_csv(os.path.join(d, f'{group_name}_{cond}_{tag}.csv'), index=False)
    print(f'  [{group_name}] saved {tag} CSVs → {d}')


def save_binned(base, drug, wash, group_name):
    """Save frame-binned (mean per bin) normalised CSVs for each bin size."""
    d = _out_dir(group_name)
    for bin_size in BIN_SIZES:
        for df, cond in zip([base, drug, wash], ['baseline', 'drug', 'washout']):
            if df.empty or 'Starting Frame' not in df.columns:
                continue
            work = df.copy()
            bs   = (work['Starting Frame'] // bin_size * bin_size).astype(int)
            work['Starting Frame'] = bs.astype(str) + '-' + (bs + bin_size).astype(str)
            gcols  = [c for c in ['data_folder','slice_num','Starting Frame'] if c in work.columns]
            ncols  = [c for c in work.columns if c not in gcols]
            binned = work.groupby(gcols, sort=True)[ncols].mean().reset_index()
            binned.to_csv(os.path.join(d, f'{group_name}_{cond}_binned{bin_size}.csv'), index=False)
    print(f'  [{group_name}] saved binned CSVs (bins: {BIN_SIZES}) → {d}')

In [12]:
# =============================================================================
# STATISTICS — BETWEEN-GROUP QUANTILE REGRESSION  (Q50 / Q75 / Q90)
# =============================================================================
#
# PURPOSE
# -------
# Test whether calcium event features differ between groups at a given
# condition (Drug or Washout).  Because fold-change distributions are
# right-skewed, quantile regression is used instead of mean-based tests:
# it estimates differences at three points of the distribution —
# Q50 (median), Q75, and Q90 — so tail-heavy effects (e.g. only the
# largest events change) are distinguishable from broad shifts.
#
# MODEL
# -----
#   feature_FC ~ C(Group)   [WT as reference; intercept = WT quantile]
#
# The coefficient for each non-WT group is the difference in the q-th
# percentile of that group relative to WT at the same quantile.
#
# FDR CORRECTION
# --------------
# Benjamini–Hochberg FDR is applied within each (quantile × comparison)
# block across all 16 features — one correction per inferential question
# (e.g. 'which features differ at Q90 between WT and IP?').
#
# NOTE: CE is excluded from quantile regression — no valid events loaded.
# =============================================================================

QUANTILES   = [0.50, 0.75, 0.90]
QREG_GROUPS = ['WT', 'AV', 'IP']          # CE excluded – no valid events
COMPARISONS = [g for g in QREG_GROUPS if g != CONTROL_GROUP]


def test_features(group_dict, condition_name='Drug'):
    """
    Between-group quantile regression at Q50 / Q75 / Q90 for all features.

    For each feature, all events from each group are pooled and a quantile
    regression model is fit with Group as the predictor (WT as reference).
    This directly answers: at quantile q, how does group X differ from WT?

    Returns a DataFrame with one row per feature × comparison × quantile:
      feature, comparison, quantile,
      wt_q, comp_q, coef, ci_lo, ci_hi,   <- quantile values and coefficient
      p_raw, p_fdr                         <- raw and BH FDR p-values

    BH FDR is applied within each (quantile × comparison) block across the
    16 features — separately for each inferential question
    (e.g. 'which features differ at Q90 between WT and IP?').
    """
    print(f'\nBetween-group quantile regression: {condition_name} | '
          f'Q50/Q75/Q90 | ref={CONTROL_GROUP}')

    rows = []

    for feat in ALL_FEATURES:
        # Build long-format DataFrame: one row per event, columns = value + Group
        frames = []
        for gname in QREG_GROUPS:
            df = group_dict.get(gname, pd.DataFrame())
            if df.empty or feat not in df.columns:
                continue
            vals = df[feat].dropna()
            if len(vals) < 5:
                continue
            frames.append(pd.DataFrame({'value': vals.values, 'Group': gname}))

        if len(frames) < 2:
            continue

        long_df = pd.concat(frames, ignore_index=True)
        long_df['Group'] = pd.Categorical(
            long_df['Group'],
            categories=[CONTROL_GROUP] + COMPARISONS,
            ordered=False
        )

        for q in QUANTILES:
            try:
                fit  = smf.quantreg('value ~ C(Group)', data=long_df).fit(
                    q=q, max_iter=2000)
                wt_q = fit.params['Intercept']
                ci   = fit.conf_int()
                for comp in COMPARISONS:
                    key = f'C(Group)[T.{comp}]'
                    if key not in fit.params:
                        continue
                    rows.append({
                        'feature':    feat,
                        'comparison': comp,
                        'quantile':   q,
                        'wt_q':       wt_q,
                        'comp_q':     wt_q + fit.params[key],
                        'coef':       fit.params[key],
                        'ci_lo':      ci.loc[key, 0],
                        'ci_hi':      ci.loc[key, 1],
                        'p_raw':      fit.pvalues[key],
                        'p_fdr':      np.nan,
                    })
            except Exception as e:
                print(f'    [warn] {feat} q={q:.2f} failed: {e}')

    if not rows:
        print('  No results computed.')
        return pd.DataFrame()

    res = pd.DataFrame(rows)

    # BH FDR within each (quantile × comparison) block
    for q in QUANTILES:
        for comp in COMPARISONS:
            mask = (res['quantile'] == q) & (res['comparison'] == comp)
            if mask.sum() == 0:
                continue
            _, padj, _, _ = multipletests(
                res.loc[mask, 'p_raw'].values, method='fdr_bh')
            res.loc[mask, 'p_fdr'] = padj

    return res


def _stars(p):
    if pd.isna(p): return ''
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'


def print_results(res):
    """
    Print between-group quantile regression results for features
    significant after FDR correction.  Shows all three quantiles for
    each significant feature × comparison pair.
    """
    if res.empty:
        return

    sig_feats = res[res['p_fdr'] < ALPHA]['feature'].unique()
    if len(sig_feats) == 0:
        print('  No features significant after FDR correction.')
        return

    hdr = (f"  {'Comparison':<10}  {'Q':>4}   {'WT_q':>6}  {'Grp_q':>6}  "
           f"{'Δ':>7}  {'95% CI':>19}  {'p_raw':>8}  {'p_FDR':>8}  Sig")
    div = '  ' + '─' * (len(hdr) - 2)

    for feat in ALL_FEATURES:
        if feat not in sig_feats:
            continue
        feat_res = res[res['feature'] == feat]
        print(f'\n  ┌─ {feat}')
        print(hdr); print(div)

        for comp in COMPARISONS:
            for q in QUANTILES:
                r = feat_res[
                    (feat_res['comparison'] == comp) &
                    (feat_res['quantile']   == q)]
                if r.empty:
                    continue
                r    = r.iloc[0]
                mark = ' ◄' if r['p_fdr'] < ALPHA else ''
                print(f"  {'WT vs ' + comp:<10}  Q{int(q*100):>2}   "
                      f"{r['wt_q']:>6.3f}  {r['comp_q']:>6.3f}  "
                      f"{r['coef']:>+7.3f}  [{r['ci_lo']:>+7.3f},{r['ci_hi']:>+7.3f}]  "
                      f"{r['p_raw']:>8.4f}  {r['p_fdr']:>8.4f}  "
                      f"{_stars(r['p_fdr'])}{mark}")

    # Summary count table
    print('\n  ── N significant features per quantile (FDR < 0.05) ──')
    print(f"  {'Comparison':<10}  {'Q50':>6}  {'Q75':>6}  {'Q90':>6}")
    for comp in COMPARISONS:
        sig_sub = res[(res['p_fdr'] < ALPHA) & (res['comparison'] == comp)]
        vals = [len(sig_sub[sig_sub['quantile'] == q]) for q in QUANTILES]
        print(f"  {'WT vs '+comp:<10}  {vals[0]:>6}  {vals[1]:>6}  {vals[2]:>6}")

In [13]:
# =============================================================================
# STATISTICS — WITHIN-GROUP QUANTILE TRAJECTORY  (Baseline → Drug → Washout)
# =============================================================================
#
# PURPOSE
# -------
# Test whether each feature changes across the three recording conditions
# (Baseline, Drug, Washout) within a single group.  This is the within-group
# complement to test_features(), which compares groups at a single condition.
#
# Critically, this reveals *where in the distribution* the drug effect sits:
#   • Only Q90 shifts → the drug enlarges extreme events but leaves the
#     typical event (median) unchanged.
#   • All three quantiles shift together → a broad, uniform effect.
#   • Q50 shifts but Q90 does not → the drug compresses the distribution.
#
# MODEL
# -----
#   feature_FC ~ C(Condition)   [Baseline as reference; intercept = Baseline quantile]
#
# All events from all slices within the group are pooled per condition.
# The coefficient for Drug (or Washout) at quantile q is the difference in
# the q-th percentile relative to Baseline.
#
# NOTE: Because all values are already expressed as fold-change relative to
# each slice's own baseline median, the Baseline intercept should be ~1.0
# for Q50.  Deviations confirm the normalisation worked.
#
# FDR CORRECTION
# --------------
# Benjamini–Hochberg FDR is applied within each (quantile × condition)
# block across all 16 features, separately for Drug and Washout.
# =============================================================================

CONDITIONS       = ['baseline', 'drug', 'washout']
COND_LABELS      = ['Baseline', 'Drug', 'Washout']
TRAJ_QUANTILES   = [0.50, 0.75, 0.90]
TRAJ_ALPHA_MAP   = {0.50: 1.0, 0.75: 0.7, 0.90: 0.5}   # line opacity per quantile
TRAJ_LS          = {0.50: '-', 0.75: '--', 0.90: ':'}   # line style per quantile


def test_within_group(group_name):
    """
    Within-group quantile regression at Q50 / Q75 / Q90 for all features.

    For each feature, all events from all slices in the group are pooled
    per condition and a quantile regression model is fit with Condition as
    the predictor (Baseline as reference).  This directly answers: at
    quantile q, how does Drug (or Washout) differ from Baseline?

    Returns a DataFrame with one row per feature × condition × quantile:
      feature, quantile, condition,
      base_q,            <- Baseline quantile (model intercept)
      comp_q,            <- Drug/Washout quantile (intercept + coefficient)
      coef,              <- difference vs Baseline at this quantile
      ci_lo, ci_hi,      <- 95% CI on the coefficient
      p_raw, p_fdr       <- raw and BH FDR p-values

    BH FDR is applied within each (quantile × condition) block across the
    16 features — separately for Drug and Washout.
    """
    print(f'  Within-group quantile regression: {group_name}')
    rows = []

    for feat in ALL_FEATURES:
        frames = []
        for cond in CONDITIONS:
            df = group_data[cond].get(group_name, pd.DataFrame())
            if df.empty or feat not in df.columns:
                continue
            vals = df[feat].dropna()
            if len(vals) < 5:
                continue
            frames.append(pd.DataFrame({'value': vals.values, 'Condition': cond}))

        if len(frames) < 2:
            continue

        long_df = pd.concat(frames, ignore_index=True)
        long_df['Condition'] = pd.Categorical(
            long_df['Condition'], categories=CONDITIONS, ordered=False)

        for q in TRAJ_QUANTILES:
            try:
                fit    = smf.quantreg('value ~ C(Condition)', data=long_df).fit(
                    q=q, max_iter=2000)
                base_q = fit.params['Intercept']   # Baseline quantile
                ci     = fit.conf_int()
                for cond in CONDITIONS[1:]:         # Drug and Washout vs Baseline
                    key = f'C(Condition)[T.{cond}]'
                    if key not in fit.params:
                        continue
                    rows.append({
                        'feature':   feat,
                        'quantile':  q,
                        'condition': cond,
                        'base_q':    base_q,
                        'comp_q':    base_q + fit.params[key],
                        'coef':      fit.params[key],
                        'ci_lo':     ci.loc[key, 0],
                        'ci_hi':     ci.loc[key, 1],
                        'p_raw':     fit.pvalues[key],
                        'p_fdr':     np.nan,
                    })
            except Exception as e:
                print(f'    [warn] {feat} q={q:.2f} failed: {e}')

    if not rows:
        return pd.DataFrame()

    res = pd.DataFrame(rows)

    # BH FDR within each (quantile × condition) block
    for q in TRAJ_QUANTILES:
        for cond in CONDITIONS[1:]:
            mask = (res['quantile'] == q) & (res['condition'] == cond)
            if mask.sum() == 0:
                continue
            _, padj, _, _ = multipletests(
                res.loc[mask, 'p_raw'].values, method='fdr_bh')
            res.loc[mask, 'p_fdr'] = padj

    return res


def print_within_group_results(res, group_name):
    """
    Print within-group quantile regression results for features significant
    after FDR correction.  Shows all quantiles for each significant
    feature × condition pair.
    """
    if res.empty:
        return

    sig_feats = res[res['p_fdr'] < ALPHA]['feature'].unique()
    print(f'\nWithin-group trajectory: {group_name} | Q50/Q75/Q90 | ref=Baseline')
    if len(sig_feats) == 0:
        print('  No features significant after FDR correction.')
        return

    hdr = (f"  {'Condition':<10}  {'Q':>4}   {'Base_q':>6}  {'Cond_q':>6}  "
           f"{'Δ':>7}  {'95% CI':>19}  {'p_raw':>8}  {'p_FDR':>8}  Sig")
    div = '  ' + '─' * (len(hdr) - 2)

    for feat in ALL_FEATURES:
        if feat not in sig_feats:
            continue
        feat_res = res[res['feature'] == feat]
        print(f'\n  ┌─ {feat}')
        print(hdr); print(div)

        for cond in CONDITIONS[1:]:
            for q in TRAJ_QUANTILES:
                r = feat_res[
                    (feat_res['condition'] == cond) &
                    (feat_res['quantile']  == q)]
                if r.empty:
                    continue
                r    = r.iloc[0]
                mark = ' ◄' if r['p_fdr'] < ALPHA else ''
                print(f"  {cond.capitalize():<10}  Q{int(q*100):>2}   "
                      f"{r['base_q']:>6.3f}  {r['comp_q']:>6.3f}  "
                      f"{r['coef']:>+7.3f}  [{r['ci_lo']:>+7.3f},{r['ci_hi']:>+7.3f}]  "
                      f"{r['p_raw']:>8.4f}  {r['p_fdr']:>8.4f}  "
                      f"{_stars(r['p_fdr'])}{mark}")

    # Summary count table
    print('\n  ── N significant features per quantile (FDR < 0.05) ──')
    print(f"  {'Condition':<10}  {'Q50':>6}  {'Q75':>6}  {'Q90':>6}")
    for cond in CONDITIONS[1:]:
        sig_sub = res[(res['p_fdr'] < ALPHA) & (res['condition'] == cond)]
        vals = [len(sig_sub[sig_sub['quantile'] == q]) for q in TRAJ_QUANTILES]
        print(f"  {cond.capitalize():<10}  {vals[0]:>6}  {vals[1]:>6}  {vals[2]:>6}")

In [14]:
# =============================================================================
# DISPLAY AND PLOTTING
# =============================================================================
#
# THREE PLOT TYPES
# ----------------
# 1. plot_group_comparison   — x-axis = groups (WT / AV / IP / CE).
#                              Directly mirrors the between-group quantile test.
#                              Run once for Drug and once for Washout.
#
# 2. plot_quantile_trajectories — x-axis = conditions (Baseline / Drug / Washout).
#                              Shows Q50, Q75, Q90 as separate lines per feature.
#                              Directly mirrors the within-group quantile test.
#                              Run once per group.
#
# 3. plot_timepoints         — x-axis = conditions, per group (legacy dot plot).
#                              No quantile lines; useful as a sanity check that
#                              normalisation worked and the drug had a visible
#                              within-group effect.
#
# SHARED VISUAL CONVENTIONS
# -------------------------
# • Dots  = per-slice medians (true biological n = number of slices).
# • Bar   = group/condition median-of-slice-medians.
# • ▲     = outlier slice beyond the 99th percentile, drawn at cap.
# • Median is used throughout (not mean) because fold-change distributions
#   are right-skewed; median is robust to extreme single-event outliers.
# • Significance annotations on dot plots use Q50 quantile regression FDR
#   (Q50 = median; consistent with the per-slice median dots).
# =============================================================================

def _slice_medians(df, feature):
    """
    Per-slice median of one feature.  Returns a 1-D array of length n_slices.

    Median is used because fold-change distributions are right-skewed;
    it is robust to extreme tail events.  The group summary bar uses the
    same statistic (median-of-slice-medians) so bar and dots are consistent.
    """
    if df.empty or feature not in df.columns:
        return np.array([])
    gcols = [c for c in ['data_folder', 'slice_num'] if c in df.columns]
    if gcols:
        return df.groupby(gcols)[feature].median().dropna().values
    return df[feature].dropna().values


def _cap_yaxis(ax, values_list, quantile=0.99, margin=0.1):
    """
    Cap the y-axis at the 99th percentile of all plotted values so that a
    single outlier slice cannot compress the rest of the data into a narrow
    band.  Returns the cap value so _draw_dots can mark clipped points as
    triangles rather than omitting them.
    """
    all_vals = np.concatenate([v for v in values_list if len(v) > 0])
    if len(all_vals) == 0:
        return None
    lo = np.nanpercentile(all_vals, 1)
    hi = np.nanpercentile(all_vals, quantile * 100)
    pad = max((hi - lo) * margin, 0.1)
    ax.set_ylim(lo - pad, hi + pad)
    return hi


def _draw_dots(ax, x, values, color, rng, y_cap=None):
    """
    Jittered per-slice median dots + overall median bar.
    Dots above y_cap are drawn as upward triangles at the cap line
    so they remain visible without distorting the axis.
    """
    if len(values) == 0:
        return
    jitter = rng.uniform(-0.15, 0.15, len(values))

    if y_cap is not None:
        in_range = values <= y_cap
        clipped  = ~in_range
        if in_range.any():
            ax.scatter(x + jitter[in_range], values[in_range],
                       color=color, s=40, alpha=0.75, zorder=3, linewidths=0)
        if clipped.any():
            ax.scatter(x + jitter[clipped],
                       np.full(clipped.sum(), y_cap * 0.98),
                       color=color, s=50, alpha=0.85, zorder=3,
                       marker='^', linewidths=0.5, edgecolors='white')
    else:
        ax.scatter(x + jitter, values, color=color, s=40, alpha=0.75,
                   zorder=3, linewidths=0)

    ax.hlines(np.median(values), x - 0.3, x + 0.3,
              colors=color, linewidth=2.5, zorder=4)


def _annotate_significance(ax, feat, res, group_order, y_top):
    """
    Draw significance brackets on a dot plot using Q50 quantile regression FDR.

    Q50 is chosen for the visual annotation because it equals the median —
    the same statistic shown by the per-slice dots and summary bars.
    Full Q50 / Q75 / Q90 results are available in the print_results table.
    """
    if res is None or res.empty:
        return
    sig = res[(res['feature'] == feat) &
              (res['quantile'] == 0.50) &
              (res['p_fdr'] < ALPHA)]
    if sig.empty:
        return

    x_map  = {g: i for i, g in enumerate(group_order)}
    step   = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.08
    base_y = y_top + step * 0.3

    for k, (_, row) in enumerate(sig.iterrows()):
        g1, g2 = CONTROL_GROUP, row['comparison']
        if g1 not in x_map or g2 not in x_map:
            continue
        x1, x2 = sorted([x_map[g1], x_map[g2]])
        bar_y  = base_y + k * step
        tip    = bar_y - step * 0.15
        ax.plot([x1, x1, x2, x2], [tip, bar_y, bar_y, tip],
                lw=1.2, color='dimgrey', zorder=5)
        ax.text((x1+x2)/2, bar_y + step*0.05, _stars(row['p_fdr']),
                ha='center', va='bottom', fontsize=9,
                color='dimgrey', fontweight='bold', zorder=6)

    needed = base_y + len(sig) * step + step * 0.5
    lo, hi = ax.get_ylim()
    if needed > hi:
        ax.set_ylim(lo, needed)


def _make_grid(n_panels, n_cols=3, panel_w=5.0, panel_h=3.8):
    """
    Create a figure and flattened axes array; delete unused panels.
    3 columns is the default — wider panels give more room for y-axis labels
    and significance brackets than a 4-column layout.
    """
    n_rows = (n_panels + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * panel_w, n_rows * panel_h),
                             constrained_layout=True)
    axes = np.array(axes).flatten()
    for j in range(n_panels, len(axes)):
        fig.delaxes(axes[j])
    return fig, axes


def plot_group_comparison(group_data, condition='drug', res=None):
    """
    Between-group dot plot: x-axis = groups (WT / AV / IP / CE),
    one panel per feature.  Run once for Drug and once for Washout.

    Directly mirrors the between-group quantile regression test.

    Dots       = per-slice medians (n = number of slices).
    Bar        = group median-of-slice-medians.
    Triangles  = outlier slices beyond the 99th percentile.
    Brackets   = Q50 quantile regression FDR (requires res= argument);
                 if None, no significance annotations are drawn.

    Full Q50 / Q75 / Q90 results are in the print_results table.

    Parameters
    ----------
    group_data : dict  — {'baseline': {group: df}, 'drug': ..., 'washout': ...}
    condition  : str   — 'drug' or 'washout'
    res        : DataFrame from test_features(), or None.
    """
    cond_dict = group_data[condition]
    rng       = np.random.default_rng(42)

    n_panels = len(ALL_FEATURES)
    fig, axes = _make_grid(n_panels)
    ann_note  = '  |  brackets = Q50 quantile regression FDR < 0.05' if res is not None else ''
    fig.suptitle(
        f'Between-group comparison – {condition.capitalize()}\n'
        f'Dots = per-slice medians  |  bar = group median'
        f'  |  ▲ = outlier beyond 99th pct{ann_note}',
        fontsize=12, fontweight='bold')

    x_pos = list(range(len(GROUP_ORDER)))

    for i, feat in enumerate(ALL_FEATURES):
        ax = axes[i]

        all_vals_list = []
        group_vals    = {}
        for gname in GROUP_ORDER:
            v = _slice_medians(cond_dict.get(gname, pd.DataFrame()), feat)
            group_vals[gname] = v
            all_vals_list.append(v)

        y_cap = _cap_yaxis(ax, all_vals_list)

        for xi, gname in enumerate(GROUP_ORDER):
            _draw_dots(ax, xi, group_vals[gname],
                       COLORS.get(gname, 'grey'), rng, y_cap=y_cap)

        ax.axhline(1.0, color='grey', linestyle=':', lw=1, alpha=0.6)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(GROUP_ORDER)
        ax.set_title(_feat_label(feat), fontsize=9)
        ax.set_ylabel('Fold Change')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

        if res is not None:
            _, y_top = ax.get_ylim()
            _annotate_significance(ax, feat, res, GROUP_ORDER, y_top)

    plt.show()


def plot_quantile_trajectories(group_name, res):
    """
    Within-group quantile trajectory plot: x-axis = conditions
    (Baseline / Drug / Washout), one panel per feature.
    Run once per group.

    Directly mirrors the within-group quantile regression test.

    Three lines per panel show how Q50 (solid), Q75 (dashed), and Q90
    (dotted) of the fold-change distribution change across conditions.
    This reveals whether the drug effect is:
      • Tail-heavy: Q90 shifts but Q50 does not.
      • Broad/uniform: all three quantiles shift together.
      • Compressive: Q50 shifts but Q90 does not.

    Shaded band = 95% CI on the Q50 coefficient (Drug and Washout vs Baseline).
    '*' markers = conditions with FDR-significant difference from Baseline.

    Parameters
    ----------
    group_name : str
    res        : DataFrame from test_within_group(), or empty DataFrame.
    """
    color = COLORS.get(group_name, 'steelblue')

    fig, axes = _make_grid(len(ALL_FEATURES))
    fig.suptitle(
        f'{group_name} – Within-group quantile trajectories\n'
        'Lines = Q50 (solid) / Q75 (dashed) / Q90 (dotted)  '
        '|  shading = Q50 95% CI  |  * = FDR < 0.05 vs Baseline',
        fontsize=11, fontweight='bold')

    for i, feat in enumerate(ALL_FEATURES):
        ax = axes[i]
        feat_res = res[res['feature'] == feat] if not res.empty else pd.DataFrame()

        for q in TRAJ_QUANTILES:
            q_vals = []
            ci_lo  = []
            ci_hi  = []

            # Baseline point: taken from the model intercept (base_q)
            base_rows = feat_res[feat_res['quantile'] == q]
            base_q = base_rows['base_q'].iloc[0] if not base_rows.empty else np.nan
            q_vals.append(base_q)
            ci_lo.append(base_q)   # no CI at reference level
            ci_hi.append(base_q)

            # Drug and Washout points
            for cond in CONDITIONS[1:]:
                r = feat_res[(feat_res['quantile'] == q) & (feat_res['condition'] == cond)]
                if r.empty:
                    q_vals.append(np.nan)
                    ci_lo.append(np.nan)
                    ci_hi.append(np.nan)
                else:
                    r = r.iloc[0]
                    q_vals.append(r['comp_q'])
                    ci_lo.append(r['comp_q'] - (r['comp_q'] - r['ci_lo']))
                    ci_hi.append(r['comp_q'] + (r['ci_hi'] - r['comp_q']))

            xs = list(range(3))
            ax.plot(xs, q_vals,
                    color=color, lw=2,
                    alpha=TRAJ_ALPHA_MAP[q],
                    linestyle=TRAJ_LS[q],
                    label=f'Q{int(q*100)}')

            # Shaded CI band for Q50 only
            if q == 0.50:
                ax.fill_between(xs, ci_lo, ci_hi, color=color, alpha=0.12)

            # FDR significance markers (* above each significant condition point)
            for xi, cond in enumerate(CONDITIONS[1:], start=1):
                r = feat_res[(feat_res['quantile'] == q) & (feat_res['condition'] == cond)]
                if not r.empty and r.iloc[0]['p_fdr'] < ALPHA:
                    y_mark = q_vals[xi]
                    if not np.isnan(y_mark):
                        ax.text(xi, y_mark, '*', ha='center', va='bottom',
                                fontsize=10, color=color,
                                fontweight='bold', alpha=TRAJ_ALPHA_MAP[q])

        ax.axhline(1.0, color='grey', linestyle=':', lw=1, alpha=0.5)
        ax.set_xticks(range(3))
        ax.set_xticklabels(COND_LABELS, fontsize=8)
        ax.set_title(_feat_label(feat), fontsize=9)
        ax.set_ylabel('Fold Change')
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)
        if i == 0:
            ax.legend(fontsize=7, loc='upper right')

    plt.show()


def plot_timepoints(group_data, group_name):
    """
    Within-group dot plot: x-axis = conditions (Baseline / Drug / Washout).
    Sanity check that normalisation worked and the drug had a visible effect.

    This plot shows raw per-slice medians without quantile lines.
    Use plot_quantile_trajectories for the quantile-regression view.

    Dots       = per-slice medians (n = number of slices).
    Bar        = condition median-of-slice-medians.
    Triangles  = outlier slices beyond the 99th percentile.
    """
    base  = group_data['baseline'].get(group_name, pd.DataFrame())
    drug  = group_data['drug'].get(group_name,     pd.DataFrame())
    wash  = group_data['washout'].get(group_name,  pd.DataFrame())
    color = COLORS.get(group_name, 'steelblue')
    rng   = np.random.default_rng(42)

    cond_dfs = [base, drug, wash]

    n_panels = len(ALL_FEATURES)
    fig, axes = _make_grid(n_panels)
    fig.suptitle(
        f'{group_name} – Within-group timepoints\n'
        'Dots = per-slice medians  |  bar = condition median'
        '  |  ▲ = outlier beyond 99th pct',
        fontsize=12, fontweight='bold')

    for i, feat in enumerate(ALL_FEATURES):
        ax = axes[i]
        all_vals_list = [_slice_medians(cdf, feat) for cdf in cond_dfs]
        y_cap = _cap_yaxis(ax, all_vals_list)
        for xi, cdf in enumerate(cond_dfs):
            _draw_dots(ax, xi, _slice_medians(cdf, feat), color, rng, y_cap=y_cap)
        ax.axhline(1.0, color='grey', linestyle=':', lw=1, alpha=0.6)
        ax.set_xticks(range(3))
        ax.set_xticklabels(COND_LABELS)
        ax.set_title(_feat_label(feat), fontsize=9)
        ax.set_ylabel('Fold Change')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    plt.show()

In [15]:
# =============================================================================
# EVENT COUNT EXPORTS
# =============================================================================

def export_frame_counts(group_name):
    """
    Frame-wise event counts for one group (reads from disk — frame-level detail
    is not stored in group_data).

    Output columns: slice | frame | baseline | drug | washout
    Saves raw counts and frame-binned variants for each BIN_SIZE.
    """
    cond_cols = ['baseline', 'drug', 'washout']
    raw_rows = []

    for bp, dp, wp, subfolder, snum in _file_paths(DATA_CONFIG[group_name]):
        slice_label = f'{subfolder}_slice{snum}'
        frames = {}
        for cond, path in zip(cond_cols, [bp, dp, wp]):
            df = _load_file(path)
            frames[cond] = (df['Starting Frame'].dropna().astype(int)
                            .value_counts().sort_index().to_dict()
                            if not df.empty and 'Starting Frame' in df.columns else {})

        all_frames = sorted({f for d in frames.values() for f in d})
        if not all_frames:
            continue

        for fr in all_frames:
            row = {'slice': slice_label, 'frame': fr, '_folder': subfolder, '_slice': snum}
            for cond in cond_cols:
                row[cond] = frames[cond].get(fr, 0)
            raw_rows.append(row)

    out_cols = ['slice', 'frame'] + cond_cols

    def _to_df(rows):
        if not rows:
            return pd.DataFrame(columns=out_cols)
        return (pd.DataFrame(rows)
                .sort_values(['_folder','_slice','frame'])
                .reset_index(drop=True)[out_cols])

    raw_df = _to_df(raw_rows)
    root = os.path.join(CWD, OUTPUT_DIR)
    os.makedirs(root, exist_ok=True)

    raw_df.to_csv(os.path.join(root, f'{group_name}_event_counts_raw.csv'), index=False)
    for bs in BIN_SIZES:
        if raw_df.empty:
            continue
        work = raw_df.copy()
        work['bin'] = (work['frame'] // bs * bs).astype(int)
        binned = (work.groupby(['slice','bin'])[cond_cols]
                  .sum(min_count=1).reset_index()
                  .sort_values(['slice','bin']).reset_index(drop=True))
        binned['frame_range'] = binned['bin'].astype(str) + '-' + (binned['bin']+bs).astype(str)
        binned.to_csv(os.path.join(root, f'{group_name}_event_counts_raw_binned{bs}.csv'), index=False)

    print(f'  [{group_name}] {len(raw_df)} frame rows | bins {BIN_SIZES}')
    return raw_df

In [16]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================

# ── 1. Load data ──────────────────────────────────────────────────────────────
print('STEP 1 – Loading data')
group_data = {'baseline': {}, 'drug': {}, 'washout': {}}

for gname in DATA_CONFIG:
    b, d, w, b_raw, d_raw, w_raw = load_group(gname)
    group_data['baseline'][gname] = b
    group_data['drug'][gname]     = d
    group_data['washout'][gname]  = w
    if not b.empty:
        save_csvs(b, d, w, gname, 'normalized')
        save_csvs(b_raw, d_raw, w_raw, gname, 'raw')
        save_binned(b, d, w, gname)
    else:
        print(f'  WARNING: no valid data for {gname}')


# ── 2. Between-group quantile regression ──────────────────────────────────────
print('\nSTEP 2 – Between-group quantile regression (Q50 / Q75 / Q90)')
res_dict = {}
for cond in ['drug', 'washout']:
    cond_dict = {g: group_data[cond][g] for g in QREG_GROUPS
                 if g in group_data[cond]}
    res = test_features(cond_dict, condition_name=cond.capitalize())
    print_results(res)
    res_dict[cond] = res


# ── 3. Within-group quantile regression ───────────────────────────────────────
print('\nSTEP 3 – Within-group quantile regression (Q50 / Q75 / Q90)')
traj_res = {}
for gname in QREG_GROUPS:
    if not group_data['drug'][gname].empty:
        traj_res[gname] = test_within_group(gname)
        print_within_group_results(traj_res[gname], gname)


# ── 4. Exports ────────────────────────────────────────────────────────────────
print('\nSTEP 4 – Exports')
for gname in DATA_CONFIG:
    export_frame_counts(gname)

print('\nDone.')

STEP 1 – Loading data
  [WT] saved normalized CSVs → /home/shree05/research/Astrocytes/Output__/WT
  [WT] saved raw CSVs → /home/shree05/research/Astrocytes/Output__/WT
  [WT] saved binned CSVs (bins: [5, 10, 20]) → /home/shree05/research/Astrocytes/Output__/WT
  [AV] saved normalized CSVs → /home/shree05/research/Astrocytes/Output__/Antagonist- Volinanserin
  [AV] saved raw CSVs → /home/shree05/research/Astrocytes/Output__/Antagonist- Volinanserin
  [AV] saved binned CSVs (bins: [5, 10, 20]) → /home/shree05/research/Astrocytes/Output__/Antagonist- Volinanserin
  [IP] saved normalized CSVs → /home/shree05/research/Astrocytes/Output__/IP3R2 cKO
  [IP] saved raw CSVs → /home/shree05/research/Astrocytes/Output__/IP3R2 cKO
  [IP] saved binned CSVs (bins: [5, 10, 20]) → /home/shree05/research/Astrocytes/Output__/IP3R2 cKO
  Skipping CE/data1/slice5: incomplete triplet

STEP 2 – Between-group quantile regression (Q50 / Q75 / Q90)

Between-group quantile regression: Drug | Q50/Q75/Q90 | ref=W